# Analyse descriptive de l'AVC et de ses facteurs de risque
# 中风与危险混杂因素的描述性分析

**Données / 数据** : `healthcare-dataset-stroke-data.csv` — 5 110 lignes / 行 × 12 colonnes / 列

> ⚠️ **中文**：这是**观察性、横断面**数据（一次快照，没有时间先后）。本 notebook 只能得出「**关联**」，不能得出「**因果**」。
> **Français** : ce sont des données **observationnelles et transversales** (un instantané, sans chronologie). On ne peut conclure qu'à des **associations**, jamais à une **causalité**.

---

## Comment utiliser ce notebook / 如何使用这个 Notebook

1. **中文**：从上到下依次运行（`Shift + Enter`）。 **Français** : exécutez les cellules de haut en bas.
2. **中文**：每一步先读说明 → **先自己预测结果** → 再运行代码。 **Français** : pour chaque étape, lisez → **prédisez le résultat** → puis exécutez.
3. **中文**：每节末尾有「你来练习」，请动手修改代码。 **Français** : chaque section se termine par « À vous de jouer » — modifiez le code vous-même.
4. **中文**：把 csv 与本文件放在**同一文件夹**。只需要 `pandas / numpy / scipy / matplotlib`（`statsmodels` 可选）。
   **Français** : placez le csv dans le **même dossier** que ce fichier. Seuls `pandas / numpy / scipy / matplotlib` sont requis (`statsmodels` est optionnel).
5. **中文**：图中文字使用法语，避免不同电脑缺少中文字体而显示为方框。 **Français** : les graphiques sont en français pour éviter les carrés « □ » si la police chinoise manque.

## Glossaire trilingue / 术语对照表

| 中文 | Français | English |
|---|---|---|
| 中风 | AVC (accident vasculaire cérébral) | stroke |
| 混杂因素 | facteur de confusion | confounder |
| 中介变量 | variable médiatrice | mediator |
| 优势比 | rapport de cotes (*odds ratio*, OR) | odds ratio |
| 95% 置信区间 | intervalle de confiance à 95 % (IC 95 %) | 95% CI |
| 分层分析 | analyse stratifiée | stratified analysis |
| 敏感性分析 | analyse de sensibilité | sensitivity analysis |
| 缺失值 / 异常值 | valeurs manquantes / aberrantes | missing / outliers |
| 横断面研究 | étude transversale | cross-sectional study |
| 反向因果 | causalité inverse | reverse causation |
| 分母 | dénominateur | denominator |
| 中位数（四分位距） | médiane (écart interquartile) | median (IQR) |

In [ ]:
# ---------------------------------------------------------------
# 0. 导入库与全局设置 / Importation des bibliothèques et réglages
# ---------------------------------------------------------------
import warnings
from pathlib import Path

import numpy as np                 # 数值计算 / calcul numérique
import pandas as pd                # 表格数据处理 / manipulation de tableaux
import matplotlib.pyplot as plt    # 画图 / graphiques
from scipy import stats            # 统计检验 / tests statistiques

# display() 在 Jupyter 中自动存在；若在普通 Python 中运行，则退化为 print
# display() existe dans Jupyter ; en Python simple on se rabat sur print
try:
    from IPython.display import display
except ImportError:
    def display(*objs):
        for o in objs:
            print(o)

warnings.filterwarnings("ignore")   # 隐藏非关键警告 / masquer les avertissements non critiques

# 表格显示格式：保留 3 位小数 / affichage des tableaux : 3 décimales
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# 图形风格：去掉上/右边框，更清爽 / style : retirer les bordures haut/droite
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
# 如需在图里显示中文，取消下一行注释（需要电脑里有这些字体之一）
# Pour afficher du chinois dans les graphiques, décommentez la ligne suivante (police requise)
# plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC"]
plt.rcParams["axes.unicode_minus"] = False

# 图片保存目录（作品集里会用到）/ dossier des figures (utile pour un portfolio)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

print("Environnement prêt ✔ / 环境就绪 ✔")

---
# Étape 0 — Définir la question avant d'écrire du code / 第 0 步：先定义问题，再写代码

**中文**
「描述性分析」回答**「是什么」**，不回答**「为什么」**。我们可以说：谁中风了、中风者有什么特征、哪些因素与中风同时出现、控制年龄后这种关联是否还在。我们**不能**说「X 导致中风」。

**Français**
L'analyse descriptive répond à **« quoi ? »**, pas à **« pourquoi ? »**. On peut décrire qui a eu un AVC, quelles caractéristiques ils partagent, quels facteurs coexistent avec l'AVC, et si l'association persiste **après contrôle de l'âge**. On **ne peut pas** affirmer « X provoque l'AVC ».

### 本项目的 4 个研究问题 / Les 4 questions de recherche

| # | 中文 | Français |
|---|---|---|
| Q1 | 中风比例随**年龄**如何变化？ | Comment la proportion d'AVC varie-t-elle avec l'**âge** ? |
| Q2 | 高血压、心脏病、高血糖、肥胖、吸烟各自与中风的关联有多强？ | Quelle est la force d'association de l'hypertension, des cardiopathies, de l'hyperglycémie, de l'obésité et du tabac avec l'AVC ? |
| Q3 | 这些关联在**控制年龄之后**是否依然存在？ | Ces associations persistent-elles **après ajustement sur l'âge** ? |
| Q4 | 危险因素**叠加**时（如高血压+心脏病），中风比例如何变化？ | Que devient la proportion d'AVC quand les facteurs de risque **s'accumulent** ? |

### ✍️ 你来练习 / À vous de jouer
**中文**：在下面的单元格里写下你对每个问题的**预测**（运行代码之前！）。这是证明「思考是你的」的第一份证据。
**Français** : dans la cellule ci-dessous, écrivez vos **prédictions** (avant d'exécuter le code !). C'est la première preuve que la réflexion est bien la vôtre.

In [ ]:
# 你的预测 —— 请在看到结果之前填写 / Vos prédictions — à remplir AVANT de voir les résultats
date_prediction = "AAAA-MM-JJ"   # 填写今天的日期 / mettez la date du jour

hypotheses = {
    "H1 (âge / 年龄)":        "在这里写下预测 / écrivez votre prédiction ici",
    "H2 (facteurs / 因素)":   "哪个因素关联最强？/ quel facteur sera le plus fortement associé ?",
    "H3 (ajustement / 调整)": "控制年龄后，哪些关联会变弱？/ quelles associations s'affaibliront après ajustement sur l'âge ?",
    "H4 (cumul / 叠加)":      "叠加危险因素会怎样？/ que se passe-t-il quand les facteurs s'accumulent ?",
}

print("Prédictions enregistrées le / 预测记录日期 :", date_prediction)
for k, v in hypotheses.items():
    print(f"  {k} → {v}")

---
# Étape 1 — Lire le fichier et comprendre chaque colonne / 第 1 步：读数据、读懂每一列

**中文**：先建立「数据字典」：每列的含义、类型、以及**不确定的地方**。不确定之处将来要写进「局限性」。
**Français** : construisez d'abord un « dictionnaire de données » : sens, type et **zones d'incertitude** de chaque colonne. Ces incertitudes iront dans la section « limites ».

In [ ]:
# ---------------------------------------------------------------
# 1.1 读取数据 / Lecture des données
# ---------------------------------------------------------------
# 依次尝试几个常见路径 / on essaie plusieurs chemins courants
CANDIDATES = [
    Path("healthcare-dataset-stroke-data.csv"),
    Path("data/healthcare-dataset-stroke-data.csv"),
    Path("/mnt/user-data/uploads/healthcare-dataset-stroke-data.csv"),
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "找不到 csv：请把它放在本 notebook 同一文件夹 / "
        "CSV introuvable : placez-le dans le même dossier que ce notebook."
    )

df_raw = pd.read_csv(DATA_PATH)   # 原始数据，永远不要直接修改它 / données brutes : ne jamais les modifier
df = df_raw.copy()                # 工作副本 / copie de travail

print("Fichier / 文件 :", DATA_PATH)
print("Dimensions / 维度 (lignes, colonnes) :", df.shape)
display(df.head())                # 看前 5 行 / premières lignes

In [ ]:
# ---------------------------------------------------------------
# 1.2 数据字典 / Dictionnaire de données
# ---------------------------------------------------------------
# 最后一列「需要留意」记录你不确定的地方 / la dernière colonne note ce dont on n'est pas sûr
data_dictionary = pd.DataFrame([
    ["id",                "整数 entier",      "唯一编号 / identifiant unique",                          "不参与分析 / non utilisé dans l'analyse"],
    ["gender",            "文本 texte",       "性别 / sexe (Male, Female, Other)",                      "'Other' 仅 1 行 / une seule ligne"],
    ["age",               "小数 décimal",     "年龄（岁）/ âge en années",                              "小数 = 婴儿 / décimales = nourrissons"],
    ["hypertension",      "0/1",              "高血压 / hypertension",                                  "是诊断史还是当前测量？未说明 / diagnostic ou mesure ? non précisé"],
    ["heart_disease",     "0/1",              "心脏病 / cardiopathie",                                  "定义未说明 / définition non précisée"],
    ["ever_married",      "Yes/No",           "是否结过婚 / déjà marié(e)",                             "很可能是年龄的替身 / probablement un substitut de l'âge"],
    ["work_type",         "文本 texte",       "职业类型 / type d'emploi",                               "'children' ≈ 未成年人 / ≈ mineurs"],
    ["Residence_type",    "Urban/Rural",      "居住地 / lieu de résidence",                             "—"],
    ["avg_glucose_level", "小数 décimal",     "平均血糖 / glycémie moyenne (mg/dL)",                    "空腹还是随机？未说明 / à jeun ou non ? non précisé"],
    ["bmi",               "小数 décimal",     "体重指数 / IMC (kg/m²)",                                 "有缺失值 / valeurs manquantes"],
    ["smoking_status",    "文本 texte",       "吸烟状态 / statut tabagique",                            "'Unknown' 含义不清 / 'Unknown' ambigu"],
    ["stroke",            "0/1",              "结局：是否中风 / issue : AVC (1) ou non (0)",            "本分析的因变量 / variable à expliquer"],
], columns=["变量 Variable", "类型 Type", "含义 Signification", "⚠ 需要留意 / À surveiller"])

display(data_dictionary)

---
# Étape 2 — Audit de la qualité des données / 第 2 步：数据质量审计

**中文**：这是新手最容易跳过、却最重要的一步。原则：**先怀疑数据，再相信结论**。
**Français** : c'est l'étape que les débutants sautent le plus souvent, et la plus importante. Principe : **douter des données avant de croire les conclusions**.

我们将检查 / Nous vérifierons :
1. 基本概况、重复、结局比例 / aperçu général, doublons, proportion d'AVC
2. 缺失值——**是随机缺失吗？** / valeurs manquantes — **sont-elles aléatoires ?**
3. `Unknown` 与儿童 / `Unknown` et enfants
4. 异常值 / valeurs aberrantes

In [ ]:
# ---------------------------------------------------------------
# 2.1 基本概况 / Aperçu général
# ---------------------------------------------------------------
print("类型 / Types de colonnes :")
print(df.dtypes.to_string())

print("\n重复的 id / Doublons d'id :", df["id"].duplicated().sum())

# 结局分布：注意类别不平衡！/ distribution de l'issue : attention au déséquilibre !
n_total  = len(df)
n_stroke = int(df["stroke"].sum())
print(f"\n中风人数 / Nombre d'AVC : {n_stroke} / {n_total} = {n_stroke / n_total * 100:.2f} %")

# 连续变量概览 / résumé des variables continues
display(df[["age", "avg_glucose_level", "bmi"]].describe().T)

In [ ]:
# ---------------------------------------------------------------
# 2.2 缺失值：BMI / Valeurs manquantes : IMC
# ---------------------------------------------------------------
print("每列缺失数 / Manquants par colonne :")
print(df.isna().sum().to_string())

# 关键问题：缺失是随机的吗？把「BMI 是否缺失」当成一个分组变量，看它与中风的关系
# Question clé : le manque est-il aléatoire ? On traite « IMC manquant ou non » comme un groupe
df["bmi_missing"] = df["bmi"].isna()

bmi_check = df.groupby("bmi_missing").agg(
    n=("stroke", "size"),
    evenements_AVC=("stroke", "sum"),
    taux_AVC_pct=("stroke", lambda s: s.mean() * 100),
    age_moyen=("age", "mean"),
)
display(bmi_check)

n_lost = int(bmi_check.loc[True, "evenements_AVC"])
print(f"\n➜ 若直接删除 BMI 缺失行，会丢掉 {n_lost} 例中风（占全部 {n_lost / n_stroke * 100:.1f}%）。")
print(f"➜ Supprimer ces lignes ferait perdre {n_lost} AVC ({n_lost / n_stroke * 100:.1f} % du total).")
print("➜ 缺失者的中风率明显更高 → 缺失【很可能不是】随机的，不能悄悄地删除或用均值填补。")
print("➜ Le taux d'AVC est bien plus élevé chez les manquants → le manque n'est très probablement PAS aléatoire.")

In [ ]:
# ---------------------------------------------------------------
# 2.3 'Unknown' 吸烟状态、儿童、work_type / 'Unknown', enfants, work_type
# ---------------------------------------------------------------
df["is_minor"] = df["age"] < 18

# (a) 各年龄组里 smoking_status 的构成（按行归一化）
# (a) composition du statut tabagique selon mineur/adulte (normalisée par ligne)
print("吸烟状态构成 / Statut tabagique (proportions par ligne) :")
display(pd.crosstab(df["is_minor"].map({True: "Mineur <18", False: "Adulte ≥18"}),
                    df["smoking_status"], normalize="index") * 100)
print("➜ 未成年人里 ~80% 是 Unknown：这是「没有询问」，与成人的 Unknown 含义不同。")
print("➜ ~80 % des mineurs sont 'Unknown' : « non demandé », différent de l'Unknown des adultes.\n")

# (b) 儿童样本 / échantillon des mineurs
print(f"未成年人 / Mineurs : {df['is_minor'].sum()} 人，其中中风 / dont AVC : {df.loc[df['is_minor'], 'stroke'].sum()}")

# (c) work_type 各类的平均年龄 —— 'children' 与 'Never_worked' 其实是「年龄」信息
# (c) âge moyen par work_type — 'children' et 'Never_worked' encodent en fait l'âge
display(df.groupby("work_type")["age"].agg(n="size", age_moyen="mean", age_max="max"))

In [ ]:
# ---------------------------------------------------------------
# 2.4 异常值与其他问题 / Valeurs aberrantes et autres problèmes
# ---------------------------------------------------------------
print("BMI > 60 :", (df["bmi"] > 60).sum(), " 人 / cas  → 可能是录入错误 / erreur de saisie possible")
print("BMI < 15 :", (df["bmi"] < 15).sum(), " 人 / cas  → 儿童 BMI 需用儿童标准 / IMC enfant : normes pédiatriques")
print("年龄 < 1 岁 / Âge < 1 an :", (df["age"] < 1).sum())
print("gender = Other :", (df["gender"] == "Other").sum(), " 行 → 单行无法分析 / une ligne : non analysable")
print("血糖 ≥ 200 / Glycémie ≥ 200 :", (df["avg_glucose_level"] >= 200).sum(),
      "（定义不明，谨慎解读 / définition floue : prudence）")

# 画分布图：看看有没有奇怪的形状 / distributions : repérer les formes étranges
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col, titre in zip(axes, ["age", "avg_glucose_level", "bmi"],
                          ["Âge (ans)", "Glycémie moyenne (mg/dL)", "IMC (kg/m²)"]):
    ax.hist(df[col].dropna(), bins=40, color="#4C72B0", edgecolor="white")
    ax.set_title(titre)
    ax.set_ylabel("Effectif")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig0_distributions.png", bbox_inches="tight")
plt.show()

### ✍️ 你来练习 / À vous de jouer
1. **中文**：`avg_glucose_level` 的分布明显有两个「峰」。你觉得可能是什么原因？（提示：混合人群？糖尿病？测量方式？）
   **Français** : la glycémie présente deux « bosses ». Quelles explications possibles ? (indice : population mixte ? diabète ? mode de mesure ?)
2. **中文**：BMI 缺失者的平均年龄更大（52 岁 vs 43 岁）。**这会如何影响**上面的结论？
   **Français** : les personnes à IMC manquant sont plus âgées (52 vs 43 ans). **Comment cela influence-t-il** la conclusion précédente ?

---
# Étape 3 — Population d'analyse et variables dérivées / 第 3 步：确定分析人群与变量处理

**中文**：每一个清洗决定都要**记录理由**（见最后的「决策日志」）。本 notebook 做出的决定：

| 决定 | 理由 |
|---|---|
| 剔除 `gender = Other`（1 行） | 单行无法估计任何量 |
| **主分析仅限 ≥18 岁成人** | 婚姻、职业、吸烟对儿童无意义；儿童仅 2 例中风，会稀释成人结果 |
| BMI 缺失：主分析用「完整案例」，并做**敏感性分析**对比 4 种处理 | 缺失很可能非随机，任何单一方法都可能有偏 |
| 年龄、BMI、血糖分组 | 便于解释；代价是损失信息，因此同时保留连续版本 |

**Français** : chaque décision de nettoyage doit être **justifiée et tracée** (voir « journal de décisions » à la fin).

| Décision | Justification |
|---|---|
| Exclure `gender = Other` (1 ligne) | Une ligne ne permet aucune estimation |
| **Analyse principale : adultes ≥ 18 ans** | Mariage, emploi, tabac n'ont pas de sens chez l'enfant ; seulement 2 AVC < 18 ans |
| IMC manquant : « cas complets » en principe + **analyse de sensibilité** avec 4 méthodes | Manque probablement non aléatoire → aucune méthode unique n'est sûre |
| Regroupements d'âge, IMC, glycémie | Faciles à interpréter ; perte d'information → on garde aussi les versions continues |

In [ ]:
# ---------------------------------------------------------------
# 3.1 构建分析数据集 / Construction du jeu de données d'analyse
# ---------------------------------------------------------------
# (1) 剔除 gender = Other / exclure gender = Other
df = df[df["gender"] != "Other"].copy()

# (2) 年龄分组（含未成年）；right=False 表示区间 [左, 右) / groupes d'âge ; right=False → intervalles [a, b[
df["age_group"] = pd.cut(
    df["age"], bins=[0, 18, 30, 40, 50, 60, 70, 80, 200], right=False,
    labels=["<18", "18-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"],
)

# (3) 用于「分层分析」的三层年龄 / trois strates d'âge pour l'analyse stratifiée
df["age_stratum"] = pd.cut(
    df["age"], bins=[0, 18, 50, 65, 200], right=False,
    labels=["<18", "18-49", "50-64", "65+"],
)

# (4) BMI 分类（WHO 成人标准）；缺失单独成类 "Manquant"
# (4) catégories d'IMC (OMS, adultes) ; les manquants forment leur propre catégorie
bmi_labels = ["Insuffisant (<18.5)", "Normal (18.5-24.9)", "Surpoids (25-29.9)", "Obésité (≥30)"]
bmi_cat = pd.cut(df["bmi"], bins=[0, 18.5, 25, 30, 200], right=False, labels=bmi_labels).astype(object)
bmi_cat = bmi_cat.where(df["bmi"].notna(), "Manquant")
df["bmi_cat"] = pd.Categorical(bmi_cat, categories=bmi_labels + ["Manquant"], ordered=True)

# (5) 血糖分组：仅为参考切点，因为血糖测量方式未说明
# (5) glycémie : seuils indicatifs (mode de mesure non précisé)
df["glucose_cat"] = pd.cut(
    df["avg_glucose_level"], bins=[0, 100, 140, 200, 1000], right=False,
    labels=["<100", "100-139", "140-199", "≥200"],
)

# (6) 主分析人群：成人 / population principale : adultes
adults = df[df["age"] >= 18].copy()

print(f"全部（剔除 Other 后） / Ensemble : {len(df)} 人，中风 / AVC : {int(df['stroke'].sum())}")
print(f"成人 / Adultes ≥18      : {len(adults)} 人，中风 / AVC : {int(adults['stroke'].sum())}"
      f"（{adults['stroke'].mean() * 100:.2f} %）")
display(adults["bmi_cat"].value_counts().sort_index().to_frame("n"))

---
# Étape 4 — Tableau 1 : décrire les deux groupes côte à côte / 第 4 步：「表 1」中风组 vs 非中风组

**中文**：医学论文的标准第一张表。
- **连续变量**（年龄、血糖、BMI）分布偏斜 → 用**中位数 [四分位距]**，检验用 **Mann-Whitney**。
- **分类变量** → **n (%)**（占该组的百分比），检验用**卡方检验**。

**Français** : le classique « Tableau 1 » des articles médicaux.
- **Variables continues** (âge, glycémie, IMC), distributions asymétriques → **médiane [écart interquartile]**, test de **Mann-Whitney**.
- **Variables catégorielles** → **n (%)** (pourcentage *dans la colonne*), test du **khi-deux**.

> ⚠️ **中文**：p 值 ≠ 重要性。样本大时，很小的差异也可能「显著」。**Français** : p-valeur ≠ importance. Sur un grand échantillon, de petites différences peuvent être « significatives ».

In [ ]:
# ---------------------------------------------------------------
# 4.1 表 1 函数 / Fonction du Tableau 1
# ---------------------------------------------------------------
def fmt_p(p):
    """把 p 值格式化 / formater la p-valeur"""
    if pd.isna(p):
        return ""
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def table1(data, cont_vars, cat_vars, group="stroke"):
    """
    data      : DataFrame
    cont_vars : 连续变量列表 / liste des variables continues
    cat_vars  : 分类变量列表 / liste des variables catégorielles
    group     : 分组变量 (0/1) / variable de groupe (0/1)
    """
    rows = []
    g0 = data[data[group] == 0]   # 非中风 / sans AVC
    g1 = data[data[group] == 1]   # 中风 / avec AVC

    # --- 连续变量：中位数 [Q1–Q3] + Mann-Whitney ---
    for v in cont_vars:
        a, b = g0[v].dropna(), g1[v].dropna()
        p = stats.mannwhitneyu(a, b).pvalue
        rows.append({
            "Variable": v, "Modalité": "médiane [Q1–Q3]",
            f"Sans AVC (n={len(g0)})": f"{a.median():.1f} [{a.quantile(.25):.1f}–{a.quantile(.75):.1f}]",
            f"AVC (n={len(g1)})":      f"{b.median():.1f} [{b.quantile(.25):.1f}–{b.quantile(.75):.1f}]",
            "p": fmt_p(p),
        })

    # --- 分类变量：n (%) + 卡方 ---
    for v in cat_vars:
        ct = pd.crosstab(data[v], data[group])
        ct = ct.loc[ct.sum(axis=1) > 0]            # 去掉空行，避免卡方出错 / retirer les lignes vides
        p = stats.chi2_contingency(ct)[1]
        for i, lvl in enumerate(ct.index):
            rows.append({
                "Variable": v if i == 0 else "", "Modalité": str(lvl),
                f"Sans AVC (n={len(g0)})": f"{ct.loc[lvl, 0]} ({ct.loc[lvl, 0] / ct[0].sum() * 100:.1f} %)",
                f"AVC (n={len(g1)})":      f"{ct.loc[lvl, 1]} ({ct.loc[lvl, 1] / ct[1].sum() * 100:.1f} %)",
                "p": fmt_p(p) if i == 0 else "",
            })
    return pd.DataFrame(rows)

t1 = table1(
    adults,
    cont_vars=["age", "avg_glucose_level", "bmi"],
    cat_vars=["gender", "hypertension", "heart_disease", "ever_married",
              "work_type", "Residence_type", "smoking_status", "bmi_cat", "glucose_cat"],
)
display(t1)

### ✍️ 你来练习 / À vous de jouer
**中文**：观察表 1：中风组和非中风组**差别最大**的 3 个变量是什么？你能不能马上判断它们各自是「危险因素」还是「年龄的影子」？（下一步会告诉你为什么这个问题很重要。）
**Français** : dans le Tableau 1, quelles 3 variables diffèrent le plus entre les groupes ? Sont-elles de vrais « facteurs de risque » ou « l'ombre de l'âge » ? (L'étape suivante montre pourquoi la question est cruciale.)

---
# Étape 5 — Analyse bivariée : comparer des **taux**, pas des effectifs / 第 5 步：双变量分析——比较「率」，不是「数量」

**中文**
- 「中风者里吸烟的有多少」≠「吸烟者里中风的有多少」——**分母不同**！我们要的是后者：**每个类别中的中风率**。
- 每个率都配上 **n** 和 **95% 置信区间（Wilson 法）**：样本越小，区间越宽，结论越不可靠。

**Français**
- « Parmi les AVC, combien fument ? » ≠ « Parmi les fumeurs, combien ont eu un AVC ? » — **le dénominateur diffère**. On veut le second : le **taux d'AVC dans chaque catégorie**.
- Chaque taux est accompagné de **n** et d'un **IC à 95 % (méthode de Wilson)** : plus l'effectif est petit, plus l'intervalle est large.

In [ ]:
# ---------------------------------------------------------------
# 5.1 通用函数：Wilson 置信区间、分类别中风率、卡方 p 值
# 5.1 Fonctions : IC de Wilson, taux par catégorie, p-valeur du khi-deux
# ---------------------------------------------------------------
def wilson_ci(k, n, alpha=0.05):
    """
    比例的 Wilson 置信区间（比简单的正态近似更稳，特别是小样本/罕见事件）
    IC de Wilson pour une proportion (plus fiable que l'approximation normale
    quand l'effectif est petit ou l'événement rare).
    k = 事件数 / nombre d'événements ; n = 总数 / effectif total
    """
    if n == 0:
        return (np.nan, np.nan)
    z = stats.norm.ppf(1 - alpha / 2)          # 1.96 对应 95% / 1,96 pour 95 %
    p = k / n
    denom  = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half   = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half


def rate_table(data, var, outcome="stroke", order=None):
    """
    每个类别的：样本数 n、事件数、中风率(%)、95% 置信区间
    Pour chaque modalité : effectif n, nb d'événements, taux (%), IC 95 %
    """
    g = data.groupby(var, observed=True)[outcome].agg(n="size", evenements="sum")
    if order is not None:
        g = g.reindex(order)                    # 指定显示顺序 / imposer l'ordre d'affichage
    g["taux_%"] = g["evenements"] / g["n"] * 100
    ci = [wilson_ci(k, n) for k, n in zip(g["evenements"], g["n"])]
    g["IC95_bas_%"]  = [c[0] * 100 for c in ci]
    g["IC95_haut_%"] = [c[1] * 100 for c in ci]
    return g


def chi2_p(data, var, outcome="stroke"):
    """卡方检验 p 值 / p-valeur du khi-deux (peu fiable si effectifs attendus < 5 → Fisher)"""
    ct = pd.crosstab(data[var], data[outcome])
    ct = ct.loc[ct.sum(axis=1) > 0]
    return stats.chi2_contingency(ct)[1]


# 逐个因素输出中风率表 / tableau des taux pour chaque facteur (adultes)
factors = ["hypertension", "heart_disease", "ever_married", "gender",
           "Residence_type", "smoking_status", "bmi_cat", "glucose_cat", "work_type"]

for v in factors:
    print(f"\n=== {v}   (khi-deux p = {fmt_p(chi2_p(adults, v))}) ===")
    display(rate_table(adults, v).round(2))

In [ ]:
# ---------------------------------------------------------------
# 5.2 图 1：各年龄段的中风率（全部年龄，含 <18）—— 本项目最重要的一张图
# 5.2 Figure 1 : taux d'AVC par tranche d'âge — le graphique clé du projet
# ---------------------------------------------------------------
age_rates = rate_table(df, "age_group")     # 注意：这里用 df（含未成年）/ ici df (mineurs inclus)
display(age_rates.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
x   = np.arange(len(age_rates))
y   = age_rates["taux_%"].values
err = [y - age_rates["IC95_bas_%"].values, age_rates["IC95_haut_%"].values - y]
ax.bar(x, y, yerr=err, capsize=4, color="#4C72B0")
for xi, yi, hi, n in zip(x, y, age_rates["IC95_haut_%"].values, age_rates["n"].values):
    ax.text(xi, hi + 0.6, f"n={n}", ha="center", fontsize=8)     # 在柱子上标注样本量 / annoter n
ax.set_xticks(x)
ax.set_xticklabels(age_rates.index)
ax.set_xlabel("Tranche d'âge")
ax.set_ylabel("Taux d'AVC (%)  ± IC 95 %")
ax.set_title("Le taux d'AVC augmente fortement avec l'âge")   # 标题直接写结论 / le titre énonce la conclusion
plt.tight_layout()
fig.savefig(FIG_DIR / "fig1_taux_par_age.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 5.3 图 2：主要因素的中风率（小多图）/ Figure 2 : taux par facteur (petits multiples)
# ---------------------------------------------------------------
def plot_rates(ax, data, var, title, order=None, color="#DD8452"):
    t = rate_table(data, var, order=order)
    y = t["taux_%"].values
    err = [y - t["IC95_bas_%"].values, t["IC95_haut_%"].values - y]
    ax.bar(range(len(t)), y, yerr=err, capsize=3, color=color)
    ax.set_xticks(range(len(t)))
    ax.set_xticklabels([str(i) for i in t.index], rotation=25, ha="right", fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("Taux d'AVC (%)")

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
plot_rates(axes[0, 0], adults, "hypertension",  "Hypertension (0=non, 1=oui)")
plot_rates(axes[0, 1], adults, "heart_disease", "Cardiopathie (0=non, 1=oui)")
plot_rates(axes[0, 2], adults, "ever_married",  "Déjà marié(e)")
plot_rates(axes[1, 0], adults, "smoking_status", "Statut tabagique",
           order=["never smoked", "formerly smoked", "smokes", "Unknown"])
plot_rates(axes[1, 1], adults, "bmi_cat", "IMC (catégories OMS)")
plot_rates(axes[1, 2], adults, "glucose_cat", "Glycémie moyenne (mg/dL)")
plt.suptitle("Taux d'AVC bruts chez les adultes (barres = IC 95 %)  —  ⚠ sans ajustement sur l'âge", y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "fig2_taux_par_facteur.png", bbox_inches="tight")
plt.show()

### ✍️ 你来练习 / À vous de jouer
**中文**：图 2 里「已婚」的中风率明显高于「未婚」。**在往下看之前**，请写下 1 个可能的解释——不是「结婚使人中风」，而是一个**能被数据检验**的假设。
**Français** : dans la Figure 2, le taux d'AVC des personnes mariées est nettement plus élevé. **Avant de continuer**, notez une explication possible — pas « le mariage provoque l'AVC », mais une hypothèse **testable avec les données**.

---
# Étape 6 — Facteurs de confusion : stratification et ajustement / 第 6 步：混杂因素——分层与调整（整个项目的灵魂）

**中文**
**混杂因素（facteur de confusion）**= 同时与「暴露」（如婚姻）和「结局」（中风）相关、且不在因果链上的第三个变量。在这份数据里，**年龄**就是最主要的混杂因素。

**Français**
Un **facteur de confusion** est une troisième variable, liée à la fois à l'« exposition » (ex. mariage) et à l'issue (AVC), et qui n'est pas sur la chaîne causale. Ici, **l'âge** est le principal facteur de confusion.

**思考步骤 / Démarche** :
1. 年龄和「暴露」有关吗？→ 6.1 / L'âge est-il lié à l'exposition ? → 6.1
2. 分层后，关联还在吗？→ 6.2、6.3 / Après stratification, l'association persiste-t-elle ? → 6.2, 6.3
3. 一次性调整多个变量 → 6.6 / Ajuster plusieurs variables à la fois → 6.6

In [ ]:
# ---------------------------------------------------------------
# 6.1 证据一：年龄与其他变量高度相关 / Preuve 1 : l'âge est lié aux autres variables
# ---------------------------------------------------------------
# 用全部人群展示这个现象 / on utilise toute la population pour montrer le phénomène
for v in ["ever_married", "hypertension", "heart_disease", "smoking_status", "work_type"]:
    print(f"\n年龄均值 / Âge moyen selon {v} :")
    display(df.groupby(v, observed=True)["age"].agg(n="size", age_moyen="mean").round(1))

print("\n➜ 已婚者比未婚者平均大 30 多岁；有高血压/心脏病者也明显更老。")
print("➜ Les personnes mariées ont plus de 30 ans de plus en moyenne ; idem pour hypertension / cardiopathie.")
print("➜ 年龄同时与「暴露」和「中风」相关 = 混杂因素的定义。")
print("➜ L'âge est lié à l'exposition ET à l'AVC = définition d'un facteur de confusion.")

In [ ]:
# ---------------------------------------------------------------
# 6.2 辅助函数：2x2 表、粗 OR、Mantel-Haenszel 合并 OR
# 6.2 Fonctions : tableau 2x2, OR brut, OR commun de Mantel-Haenszel
# ---------------------------------------------------------------
def two_by_two(sub, exposure, exposed_value, outcome="stroke"):
    """
    返回 2x2 数组 [[a, b], [c, d]]
    a = 暴露且中风, b = 暴露未中风, c = 未暴露且中风, d = 未暴露未中风
    Retourne [[a, b], [c, d]] :
    a = exposé & AVC, b = exposé & pas d'AVC, c = non exposé & AVC, d = non exposé & pas d'AVC
    """
    e = sub[exposure] == exposed_value
    s = sub[outcome] == 1
    return np.array([[(e & s).sum(),  (e & ~s).sum()],
                     [(~e & s).sum(), (~e & ~s).sum()]], dtype=float)


def crude_or(table, alpha=0.05):
    """粗 OR + Woolf 置信区间 / OR brut + IC de Woolf"""
    (a, b), (c, d) = table
    or_ = (a * d) / (b * c)
    se  = np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    z   = stats.norm.ppf(1 - alpha / 2)
    return or_, np.exp(np.log(or_) - z * se), np.exp(np.log(or_) + z * se)


def mantel_haenszel(tables, alpha=0.05):
    """
    Mantel-Haenszel 合并 OR：把每一层里的 OR 加权合并 → 已「控制」分层变量
    置信区间用 Robins-Breslow-Greenland 方差
    OR commun de Mantel-Haenszel : combine les OR de chaque strate → l'effet de la
    variable de stratification est « contrôlé ». IC par la variance de Robins-Breslow-Greenland.
    """
    R = S = 0.0
    sum_PR = sum_PS_QR = sum_QS = 0.0
    for t in tables:
        (a, b), (c, d) = t
        n = a + b + c + d
        r, s = a * d / n, b * c / n
        P, Q = (a + d) / n, (b + c) / n
        R += r; S += s
        sum_PR += P * r
        sum_PS_QR += P * s + Q * r
        sum_QS += Q * s
    or_mh = R / S
    var = sum_PR / (2 * R**2) + sum_PS_QR / (2 * R * S) + sum_QS / (2 * S**2)
    z = stats.norm.ppf(1 - alpha / 2)
    return or_mh, np.exp(np.log(or_mh) - z * np.sqrt(var)), np.exp(np.log(or_mh) + z * np.sqrt(var))

print("Fonctions prêtes ✔ / 函数就绪 ✔")

In [ ]:
# ---------------------------------------------------------------
# 6.3 经典例子：已婚 vs 未婚——粗看 vs 同龄层内比较（仅成人）
# 6.3 Exemple classique : marié vs non marié — brut vs à âge comparable (adultes)
# ---------------------------------------------------------------
# 注意：这里的分段边界与我在对话里手算的略有不同（[a,b) vs (a,b]），数字会有细微差别，结论一致。
# Remarque : les bornes des strates diffèrent légèrement de mon calcul dans la conversation
# ([a,b[ vs ]a,b]) ; les chiffres varient un peu, la conclusion est la même.

# (1) 粗率 / taux bruts
brut = rate_table(adults, "ever_married")
print("① 粗率（未控制年龄）/ Taux bruts (sans contrôle de l'âge) :")
display(brut.round(2))

# (2) 分层后各层的率 / taux dans chaque strate d'âge
strat = (adults.groupby(["age_stratum", "ever_married"], observed=True)["stroke"]
               .agg(n="size", evenements="sum"))
strat["taux_%"] = strat["evenements"] / strat["n"] * 100
print("\n② 分层后（同龄人内比较）/ Après stratification (à âge comparable) :")
display(strat.round(2))

# (3) 粗 OR vs Mantel-Haenszel 调整 OR / OR brut vs OR ajusté (MH)
adult_strata = [s for s in ["18-49", "50-64", "65+"]]
tables = [two_by_two(adults[adults["age_stratum"] == s], "ever_married", "Yes") for s in adult_strata]
or_c,  lo_c,  hi_c  = crude_or(two_by_two(adults, "ever_married", "Yes"))
or_mh, lo_mh, hi_mh = mantel_haenszel(tables)

print(f"\n③ OR brut / crude OR            = {or_c:.2f}  [IC 95 % {lo_c:.2f}–{hi_c:.2f}]")
print(f"   OR ajusté sur l'âge (MH) / 按年龄调整 = {or_mh:.2f}  [IC 95 % {lo_mh:.2f}–{hi_mh:.2f}]")
print("\n➜ 中文：调整年龄后，OR 大幅向 1 靠拢——婚姻的「效应」大部分是年龄的影子。")
print("➜ Français : après ajustement sur l'âge, l'OR se rapproche fortement de 1 — l'« effet » du mariage est surtout l'ombre de l'âge.")
print("➜ ⚠ 65+ 未婚者样本很小，该层的率不稳定 / strate 65+ non mariés : effectif faible, taux instable.")

In [ ]:
# ---------------------------------------------------------------
# 6.4 图 3：粗看 vs 分层后（一张图讲清混杂）
# 6.4 Figure 3 : brut vs stratifié (une image pour expliquer la confusion)
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.2))
labels = ["Toutes\ntranches", "18-49", "50-64", "65+"]
vals_yes, vals_no = [brut.loc["Yes", "taux_%"]], [brut.loc["No", "taux_%"]]
for s in ["18-49", "50-64", "65+"]:
    vals_yes.append(strat.loc[(s, "Yes"), "taux_%"])
    vals_no.append(strat.loc[(s, "No"), "taux_%"])

x = np.arange(len(labels)); w = 0.38
ax.bar(x - w/2, vals_yes, w, label="Marié(e)",     color="#C44E52")
ax.bar(x + w/2, vals_no,  w, label="Non marié(e)", color="#8172B2")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Taux d'AVC (%)")
ax.set_title("L'écart « mariés vs non mariés » s'estompe à âge comparable")
ax.legend(frameon=False)
plt.tight_layout()
fig.savefig(FIG_DIR / "fig3_confusion_mariage.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 6.5 反直觉现象：为什么「曾经吸烟」的中风率比「正在吸烟」还高？
# 6.5 Un résultat contre-intuitif : pourquoi les ex-fumeurs ont-ils un taux plus élevé que les fumeurs ?
# ---------------------------------------------------------------
# 原则：遇到反常结果，先怀疑「数据与混杂」，而不是怀疑医学常识
# Principe : devant un résultat étrange, douter d'abord des données et de la confusion, pas de la médecine.
order_smk = ["never smoked", "formerly smoked", "smokes", "Unknown"]

print("① 粗率 / Taux bruts (adultes) :")
display(rate_table(adults, "smoking_status", order=order_smk).round(2))

print("\n② 各组平均年龄（假设 A：前吸烟者更老）/ Âge moyen par groupe (hypothèse A : les ex-fumeurs sont plus âgés) :")
display(adults.groupby("smoking_status")["age"].mean().reindex(order_smk).round(1).to_frame("age_moyen"))

print("\n③ 分年龄层的中风率(%)（看差距是否缩小）/ Taux (%) par strate d'âge (l'écart diminue-t-il ?) :")
pivot = (adults.groupby(["age_stratum", "smoking_status"], observed=True)["stroke"]
               .mean().mul(100).unstack("smoking_status")[order_smk])
display(pivot.round(2))

print("\n④ 各层样本量 / Effectifs par strate :")
display(adults.groupby(["age_stratum", "smoking_status"], observed=True).size().unstack("smoking_status")[order_smk])

print("""
可检验的假设 / Hypothèses testables :
  A. 年龄混杂：前吸烟者更老                     / Confusion par l'âge : ex-fumeurs plus âgés
  B. 病后戒烟：因健康问题才戒烟（反向因果）      / « Sick-quitter » : arrêt du tabac APRÈS un problème de santé (causalité inverse)
  C. Unknown 组的构成不同                        / La catégorie Unknown a une composition différente
⚠ 本数据没有戒烟原因，B 无法直接验证——这应写进局限性。
⚠ Les données ne contiennent pas la raison de l'arrêt : B n'est pas vérifiable directement → à noter dans les limites.
""")

### ✍️ 你来练习 / À vous de jouer
**中文**：看 ③ 的分层表：分年龄层后，「前吸烟」的优势是否消失了？是全部消失还是部分？哪一层样本太小、不能下结论？
**Français** : regardez le tableau ③ : après stratification, la surcote des ex-fumeurs disparaît-elle ? Totalement ou partiellement ? Quelle strate est trop petite pour conclure ?

---
### 🧠 混杂因素 vs 中介变量 / Facteur de confusion vs variable médiatrice

**中文**：并不是「变量放得越多越好」。
- **混杂因素**（如年龄）：**应该**调整。
- **中介变量**：处在因果链**中间**。例如 `BMI → 高血压 → 中风`：若你想研究 BMI 的作用，又去「调整高血压」，就把 BMI 通过高血压起作用的那部分**抹掉**了。
- 该不该调整，取决于**你的问题**，而不是软件的默认设置。建议先在纸上画一张因果图（DAG）。

**Français** : ajouter des variables n'est pas toujours mieux.
- **Facteur de confusion** (ex. âge) : il **faut** ajuster.
- **Variable médiatrice** : elle est **au milieu** de la chaîne causale. Ex. `IMC → hypertension → AVC` : si l'on étudie l'effet de l'IMC et qu'on ajuste sur l'hypertension, on **supprime** la part de l'effet qui passe par l'hypertension.
- Ajuster ou non dépend de **votre question**, pas des réglages par défaut du logiciel. Dessinez d'abord un graphe causal (DAG) sur papier.

In [ ]:
# ---------------------------------------------------------------
# 6.6 多因素逻辑回归：同时调整多个变量，得到「调整后 OR」
# 6.6 Régression logistique multivariée : OR ajustés
# ---------------------------------------------------------------
# OR（优势比）解读：OR = 2 表示该因素组的「中风的 odds」是参照组的 2 倍；OR = 1 表示无关联。
# 注意：OR ≠ 风险比；当事件罕见时二者接近，但事件不罕见时 OR 会夸大风险比。
# Lecture de l'OR : OR = 2 → les « cotes » d'AVC sont doublées par rapport à la référence ; OR = 1 → pas d'association.
# Attention : OR ≠ risque relatif ; proches quand l'événement est rare, l'OR exagère sinon.

def logit_fit(X, y, max_iter=100, tol=1e-9):
    """
    用牛顿法(IRLS)自己实现逻辑回归 —— 不依赖 statsmodels，也让你看清它在做什么。
    Régression logistique par Newton-Raphson (IRLS), sans statsmodels : pour voir ce qui se passe.
    """
    X = np.asarray(X, float); y = np.asarray(y, float)
    beta = np.zeros(X.shape[1])
    for _ in range(max_iter):
        p    = 1 / (1 + np.exp(-(X @ beta)))         # 预测概率 / probabilités prédites
        grad = X.T @ (y - p)                          # 梯度 / gradient
        H    = X.T @ (X * (p * (1 - p))[:, None])     # 海森矩阵 / hessienne
        step = np.linalg.solve(H, grad)
        beta = beta + step
        if np.max(np.abs(step)) < tol:                # 收敛 / convergence
            break
    p   = 1 / (1 + np.exp(-(X @ beta)))
    H   = X.T @ (X * (p * (1 - p))[:, None])
    se  = np.sqrt(np.diag(np.linalg.inv(H)))          # 标准误 / erreurs-types
    z   = beta / se
    pv  = 2 * (1 - stats.norm.cdf(np.abs(z)))         # Wald 检验 / test de Wald
    return beta, se, pv


def design_matrix(data, numeric, categorical):
    """
    构造设计矩阵：截距 + 数值变量 + 哑变量（去掉参照类）
    Matrice de design : constante + variables numériques + indicatrices (modalité de référence retirée)
    categorical = {变量名: 参照类别} / {variable: modalité de référence}
    """
    parts = [pd.Series(1.0, index=data.index, name="const")]
    for v in numeric:
        parts.append(data[v].astype(float).rename(v))
    for v, ref in categorical.items():
        d = pd.get_dummies(data[v].astype(str)).astype(float)
        d = d.drop(columns=str(ref))                                    # 去掉参照类 / retirer la référence
        d.columns = [f"{v}: {c} (réf. {ref})" for c in d.columns]
        parts.append(d)
    return pd.concat(parts, axis=1)


def fit_or(data, numeric, categorical):
    """拟合并返回 OR 表（只用完整案例）/ ajuste et renvoie le tableau des OR (cas complets uniquement)"""
    used = numeric + list(categorical.keys()) + ["stroke"]
    d = data.dropna(subset=used)                                        # 完整案例 / cas complets
    X = design_matrix(d, numeric, categorical)
    beta, se, pv = logit_fit(X.values, d["stroke"].values)
    out = pd.DataFrame({
        "OR": np.exp(beta), "IC95_bas": np.exp(beta - 1.96 * se),
        "IC95_haut": np.exp(beta + 1.96 * se), "p": pv,
    }, index=X.columns).drop(index="const")
    return out, len(d), int(d["stroke"].sum())


# 建模数据：把连续变量换成更有意义的单位 / unités plus parlantes pour les variables continues
model_df = adults.copy()
model_df["age10"]     = model_df["age"] / 10                  # 每增加 10 岁 / par tranche de 10 ans
model_df["glucose10"] = model_df["avg_glucose_level"] / 10    # 每增加 10 mg/dL / par 10 mg/dL
model_df["bmi5"]      = model_df["bmi"] / 5                   # 每增加 5 kg/m² / par 5 kg/m²

NUMERIC = ["age10", "hypertension", "heart_disease", "glucose10", "bmi5"]
CATEG   = {"gender": "Female", "ever_married": "No",
           "smoking_status": "never smoked", "Residence_type": "Rural"}

# ① 粗 OR：每个变量单独放入模型 / OR bruts : une variable à la fois
crude_parts = []
for v in NUMERIC:                                   # 数值变量逐个拟合 / variables numériques une par une
    t, _, _ = fit_or(model_df, [v], {})
    crude_parts.append(t)
for v, ref in CATEG.items():                        # 分类变量逐个拟合 / variables catégorielles une par une
    t, _, _ = fit_or(model_df, [], {v: ref})
    crude_parts.append(t)
crude = pd.concat(crude_parts)

# ② 调整 OR：所有变量一起放入 / OR ajustés : toutes les variables ensemble
adj, n_used, ev_used = fit_or(model_df, NUMERIC, CATEG)

compare = crude[["OR"]].join(adj, lsuffix="_brut", rsuffix="_ajusté")
compare = compare.rename(columns={"OR_brut": "OR brut / 粗", "OR_ajusté": "OR ajusté / 调整后"})
compare["p"] = compare["p"].map(fmt_p)              # 用 <0.001 代替 0.000 / afficher <0.001 au lieu de 0.000
print(f"完整案例 / Cas complets : n = {n_used}，中风 / AVC = {ev_used}")
print("（粗 OR 与调整 OR 使用同一批完整案例，才可比 / même échantillon pour brut et ajusté, pour être comparables）")
display(compare.round(3))

In [ ]:
# ---------------------------------------------------------------
# 6.7 图 4：森林图（粗 OR vs 调整后 OR）/ Figure 4 : forest plot (OR bruts vs ajustés)
# ---------------------------------------------------------------
# 对数坐标：OR 是比值，对称地看 0.5 和 2 才合理 / échelle log : 0,5 et 2 sont symétriques autour de 1
fig, ax = plt.subplots(figsize=(9, 5.2))
names = list(adj.index)[::-1]                     # 反转使第一个变量在最上面 / inverser pour avoir le 1er en haut
ypos  = np.arange(len(names))

# 调整后 OR（带置信区间）/ OR ajustés avec IC
ax.errorbar(adj.loc[names, "OR"], ypos + 0.13,
            xerr=[adj.loc[names, "OR"] - adj.loc[names, "IC95_bas"],
                  adj.loc[names, "IC95_haut"] - adj.loc[names, "OR"]],
            fmt="o", color="#C44E52", capsize=3, label="OR ajusté (multivarié)")
# 粗 OR（仅点）/ OR bruts (points seuls)
ax.plot(crude.loc[names, "OR"], ypos - 0.13, "s", color="#7F7F7F", label="OR brut")

ax.axvline(1, color="black", lw=1, ls="--")       # OR = 1：无关联 / pas d'association
ax.set_xscale("log")
ax.set_xticks([0.5, 1, 2, 4]); ax.set_xticklabels(["0.5", "1", "2", "4"])   # 用普通数字刻度 / graduations lisibles
ax.minorticks_off()
ax.set_yticks(ypos); ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("Odds ratio (échelle logarithmique)")
ax.set_title("Après ajustement, l'âge reste le facteur le plus net ; plusieurs OR bruts s'atténuent")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig4_forest_plot.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 6.8（可选）用 statsmodels 交叉验证我们的手写回归
# 6.8 (Optionnel) Vérification croisée avec statsmodels
# ---------------------------------------------------------------
# 好习惯：用第二种方法重复关键结果。如果没装 statsmodels，本单元格会自动跳过。
# Bonne habitude : refaire le résultat clé par une 2e méthode. Cellule ignorée si statsmodels est absent.
try:
    import statsmodels.formula.api as smf
    f = ("stroke ~ age10 + hypertension + heart_disease + glucose10 + bmi5"
         " + C(gender, Treatment('Female')) + C(ever_married, Treatment('No'))"
         " + C(smoking_status, Treatment('never smoked')) + C(Residence_type, Treatment('Rural'))")
    m_sm = smf.logit(f, data=model_df).fit(disp=False)
    or_sm = np.exp(m_sm.params.drop("Intercept"))
    same = np.allclose(np.sort(or_sm.values), np.sort(adj["OR"].values), rtol=1e-3)
    print("与 statsmodels 的 OR 一致吗？/ OR identiques à statsmodels ? →", same)
except ImportError:
    print("statsmodels 未安装，已跳过（pip install statsmodels）/ statsmodels absent : cellule ignorée")
except Exception as e:
    print("交叉验证出错 / Erreur de vérification :", e)

In [ ]:
# ---------------------------------------------------------------
# 6.9 危险因素的叠加（Q4）/ Cumul des facteurs de risque (Q4)
# ---------------------------------------------------------------
# n_risque = 高血压 + 心脏病 的个数(0/1/2)；与年龄组合看 / nombre de facteurs (0/1/2), croisé avec l'âge
adults["n_risque"] = adults["hypertension"] + adults["heart_disease"]
adults["age_band"] = np.where(adults["age"] >= 65, "65+", "18-64")

cumul = adults.groupby(["age_band", "n_risque"]).agg(n=("stroke", "size"), evenements=("stroke", "sum"))
cumul["taux_%"] = cumul["evenements"] / cumul["n"] * 100
ci = [wilson_ci(k, n) for k, n in zip(cumul["evenements"], cumul["n"])]
cumul["IC95_bas_%"]  = [c[0] * 100 for c in ci]
cumul["IC95_haut_%"] = [c[1] * 100 for c in ci]
display(cumul.round(2))

print("➜ 中文：注意 n 很小的格子（如 18-64 岁且两个因素都有），置信区间很宽，别过度解读。")
print("➜ Français : attention aux cases à petit n (ex. 18-64 ans avec 2 facteurs) : IC très larges, ne pas sur-interpréter.")

---
# Étape 6bis — Analyse de sensibilité : le résultat dépend-il de mes choix ? / 敏感性分析：结论是否依赖我的处理方式？

**中文**：BMI 缺失很可能非随机，所以**没有一种处理方法是「正确」的**。我们做 4 种处理，比较关键 OR 是否稳定：结论若在各种合理处理下都成立，才可信。

**Français** : le manque d'IMC étant probablement non aléatoire, **aucune méthode n'est « la bonne »**. Nous en comparons 4 : une conclusion n'est crédible que si elle résiste à plusieurs traitements raisonnables.

| # | 方法 / Méthode |
|---|---|
| 1 | 完整案例：删除 BMI 缺失行 / cas complets : supprimer les lignes sans IMC |
| 2 | 用全体中位数填补 / imputation par la médiane globale |
| 3 | 按年龄组的中位数填补 / imputation par la médiane du groupe d'âge |
| 4 | 中位数填补 + 「BMI 缺失」指示变量 / médiane + indicateur « IMC manquant » |

*(进阶 / Avancé : 多重填补 MICE / imputation multiple MICE)*

In [ ]:
# ---------------------------------------------------------------
# 6bis.1 四种 BMI 处理 → 重新拟合 → 比较关键 OR
# 6bis.1 Quatre traitements de l'IMC → refit → comparer les OR clés
# ---------------------------------------------------------------
results = {}

# 方法 1：完整案例 / cas complets
d1 = model_df.copy()
results["1. Cas complets"] = fit_or(d1, NUMERIC, CATEG)

# 方法 2：全体中位数 / médiane globale
d2 = model_df.copy()
d2["bmi5"] = d2["bmi5"].fillna(d2["bmi5"].median())
results["2. Médiane globale"] = fit_or(d2, NUMERIC, CATEG)

# 方法 3：按年龄组中位数 / médiane par groupe d'âge
d3 = model_df.copy()
d3["bmi5"] = d3.groupby("age_group", observed=True)["bmi5"].transform(lambda s: s.fillna(s.median()))
results["3. Médiane par âge"] = fit_or(d3, NUMERIC, CATEG)

# 方法 4：中位数 + 缺失指示变量 / médiane + indicateur de manque
d4 = model_df.copy()
d4["bmi_manquant"] = d4["bmi"].isna().astype(int)
d4["bmi5"] = d4["bmi5"].fillna(d4["bmi5"].median())
results["4. Médiane + indicateur"] = fit_or(d4, NUMERIC + ["bmi_manquant"], CATEG)

# 汇总表：每种方法的关键 OR / tableau de synthèse
key_terms = ["age10", "hypertension", "heart_disease", "glucose10", "bmi5"]
summary = pd.DataFrame({name: tab.loc[key_terms, "OR"] for name, (tab, n, ev) in results.items()})
print("关键变量的调整后 OR / OR ajustés des variables clés :")
display(summary.round(3))

# 样本量与事件数单独显示（整数）/ effectifs et événements affichés à part (entiers)
sizes = pd.DataFrame({name: {"n (échantillon)": n, "AVC (événements)": ev}
                      for name, (tab, n, ev) in results.items()}).astype(int)
print("\n各方法使用的样本 / Échantillon utilisé par chaque méthode :")
display(sizes)
print("➜ 方法 1 少了 181 人(含 39 例中风)；方法 2-4 保留全部成人 / La méthode 1 perd 181 adultes (dont 39 AVC) ; 2-4 gardent tous les adultes.")

# 方法 4 的「缺失指示变量」本身的 OR：缺失者是否更容易中风？
# OR de l'indicateur de manque lui-même : les manquants ont-ils plus d'AVC ?
print("\n方法 4 中「BMI 缺失」的 OR / OR de l'indicateur « IMC manquant » :",
      round(results["4. Médiane + indicateur"][0].loc["bmi_manquant", "OR"], 2))

print("\n➜ 中文：先看 age10、hypertension、heart_disease 的 OR 在 4 种处理下是否相近；再看 bmi5 是否稳定。")
print("➜ Français : vérifiez d'abord si les OR de age10, hypertension, heart_disease sont proches d'une méthode à l'autre ; puis ceux de bmi5.")

### ✍️ 你来练习 / À vous de jouer
1. **中文**：哪些 OR 在 4 种方法下几乎不变？哪个变化最大？你会如何在报告里描述「结论的稳健性」？
   **Français** : quels OR restent quasi identiques ? lequel varie le plus ? Comment décrire la « robustesse » dans le rapport ?
2. **中文**：把模型里的 `heart_disease` 和 `hypertension` 去掉，再看 `age10`、`glucose10`、`bmi5` 的 OR 有什么变化？这与「中介变量」的讨论有什么关系？
   **Français** : retirez `heart_disease` et `hypertension` du modèle : comment changent les OR de `age10`, `glucose10`, `bmi5` ? Quel rapport avec la discussion sur les variables médiatrices ?

---
# Étape 7 — Visualisation : principes / 第 7 步：可视化原则

**中文**（对照你已生成的 4 张图 `figures/` 文件夹）
1. **一张图只讲一件事**；标题写**结论**，不是变量名（如「年龄越大中风率越高」）。
2. **永远标注 n**；率要配置信区间。
3. 分层对比图（图 3）与森林图（图 4）是本项目最有「分析师思维」的两张。
4. 避免默认配色与 3D、饼图；比较用并排柱或点图。
5. 图中的文字要让**不懂统计的人**也能读懂。

**Français** (à comparer avec vos 4 figures dans le dossier `figures/`)
1. **Une figure = un message** ; le titre énonce la **conclusion**, pas le nom de la variable.
2. **Toujours afficher n** ; les taux avec leur intervalle de confiance.
3. La figure stratifiée (Fig. 3) et le forest plot (Fig. 4) sont les plus révélatrices d'une « pensée d'analyste ».
4. Éviter couleurs par défaut, 3D, camemberts ; comparer avec des barres ou des points côte à côte.
5. Le texte doit être lisible par **quelqu'un qui ne connaît pas les statistiques**.

---
# Étape 8 — Conclusions, limites, journal de décisions / 第 8 步：结论、局限性、决策日志

## 8.1 结论模板（请你用自己的话完成）/ Modèle de conclusion (à compléter avec vos mots)

**中文**：用「关联」，不用「导致」。每条结论都要同时写出：**发现 + 证据（数字）+ 可信度 + 限制**。
**Français** : dites « association », jamais « cause ». Chaque conclusion : **constat + preuve chiffrée + niveau de confiance + limite**.

1. **年龄 / Âge** : 中风率从 ___% (___岁) 上升到 ___% (___岁)；调整后每增加 10 岁 OR = ___。
   Taux d'AVC de ___ % (___ ans) à ___ % (___ ans) ; OR ajusté par 10 ans = ___.
2. **高血压 / 心脏病 / Hypertension / cardiopathie** : 粗 OR ___ → 调整后 OR ___（变弱 / 稳定？）。
3. **婚姻 / Mariage** : 粗 OR ___ → 年龄调整后 OR ___ → 解释：年龄的替身。
4. **吸烟 / Tabac** : 反直觉发现 ___；我检验了 ___；无法检验的是 ___。
5. **BMI 缺失 / IMC manquant** : 4 种处理下结论 [稳定 / 不稳定 / stable / instable]。

## 8.2 局限性 / Limites (预填，请补充 / pré-rempli, à compléter)

- **横断面观察数据**：无时间先后，只能谈关联；存在反向因果的可能。
  *Données observationnelles transversales : pas de chronologie, associations seulement ; causalité inverse possible.*
- **样本很可能不是随机人群样本**：中风率 ≈ 5%（成人 ≈ 5.8%）远高于一般人群，不能用来估计患病率。
  *Échantillon probablement non aléatoire : le taux d'AVC (~5 %) est bien supérieur à celui de la population générale → aucune estimation de prévalence.*
- **事件数少**：仅 249 例中风，许多亚组事件极少，置信区间宽。
  *Peu d'événements : 249 AVC seulement ; nombreux sous-groupes avec très peu d'événements, IC larges.*
- **变量定义不明**：血糖（空腹？）、高血压（诊断？）、吸烟 `Unknown`。
  *Définitions floues : glycémie (à jeun ?), hypertension (diagnostiquée ?), tabac `Unknown`.*
- **BMI 缺失很可能非随机**：任何处理都可能有偏（见敏感性分析）。
  *IMC manquant probablement non aléatoire : tout traitement peut biaiser (voir analyse de sensibilité).*
- **缺少关键变量**：家族史、用药、饮食、运动、吸烟量（包年）、中风类型与发病时间。
  *Variables absentes : antécédents familiaux, traitements, alimentation, activité, paquets-années, type et date de l'AVC.*
- **该数据集的来源与采集方式未充分说明**（应谨慎对待外推）。
  *Provenance et mode de collecte insuffisamment documentés : prudence pour toute généralisation.*

## 8.3 决策日志（模板）/ Journal de décisions (modèle)

| # | 决定 / Décision | 备选 / Alternatives | 理由 / Raison | 是否验证影响？/ Impact vérifié ? |
|---|---|---|---|---|
| 1 | 剔除 `gender=Other` | 保留为单独类别 | 仅 1 行 | 否（影响可忽略） |
| 2 | 主分析限成人 ≥18 | 全人群 | 儿童变量无意义、事件极少 | 部分（图 1 显示儿童几乎为 0） |
| 3 | BMI 缺失：完整案例为主 | 均值/中位数/指示变量 | 缺失很可能非随机 | ✅ 见 6bis |
| 4 | 年龄分 3 层做分层 | 更细的层 | 各层要有足够事件数 | 可自行尝试 |
| 5 | OR 用连续变量的「每 10 岁 / 10 mg/dL / 5 kg/m²」 | 分箱 | 保留信息，易解释 | 可用分箱版本对照 |
| 6 | | | | |

## 8.4 「审计 AI」清单——把它变成你的作品集亮点 / Liste d'« audit de l'IA » — votre atout portfolio

**中文**：让 AI 生成一版「标准流程」，然后逐条检查并**记录你发现的问题和你的修正**：
**Français** : demandez à une IA un pipeline « standard », puis vérifiez chaque point et **documentez ce que vous avez trouvé et corrigé** :

- [ ] 是否悄悄删除了 BMI 缺失行？ / A-t-elle supprimé silencieusement les lignes sans IMC ?
- [ ] 是否对儿童套用成人 BMI 标准？ / A-t-elle appliqué les seuils d'IMC adultes aux enfants ?
- [ ] 是否把 `Unknown` 当成一个普通吸烟类别？ / A-t-elle traité `Unknown` comme une modalité ordinaire ?
- [ ] 是否比较「数量」而不是「率」？ / A-t-elle comparé des effectifs plutôt que des taux ?
- [ ] 是否做了分层/调整？是否把中介变量当混杂因素？ / Y a-t-il stratification/ajustement ? Une médiatrice a-t-elle été traitée comme confondant ?
- [ ] 结论里有没有出现「导致 / cause / provoque」？ / Les mots « cause / provoque / entraîne » apparaissent-ils dans les conclusions ?
- [ ] 是否用机器学习「准确率」掩盖了 4.9% 的类别不平衡？ / A-t-elle masqué le déséquilibre (4,9 %) par une « accuracy » flatteuse ?

## 8.5 下一步 / Pistes suivantes

1. **中文**：给一页「非技术读者摘要」（不出现 OR 与 p 值）。 **Français** : rédigez un résumé d'une page pour non-spécialistes (sans OR ni p-valeur).
2. **中文**：（进阶）预测模型：用 AUC、PR 曲线、校准；说明年龄单变量已能达到多少。 **Français** : (avancé) modèle prédictif : AUC, courbe PR, calibration ; montrez ce que l'âge seul permet déjà.
3. **中文**：（进阶）多重填补（MICE）与 bootstrap 置信区间。 **Français** : (avancé) imputation multiple (MICE) et IC par bootstrap.